# Explorando o LangGraph

Até agora trabalhamos com cadeias de execução lineares, em que o fluxo é fixo e previsível: o dado entra, passa por um modelo e gera uma saída. Agora, vamos mudar para uma arquitetura não linear, que passa a ser baseada em grafos. Vamos começar a explorar o **LangGraph**, uma ferramenta fundamental para a criação de aplicações que não dependem apenas de cadeias lineares, e são capazes de tomar decisões sobre qual caminho seguir ou qual especialização utilizar em tempo de execução.

Com o LangGraph é possível ter:

- Execução paralela: o sistema pode processar informações de formas distintas simultaneamente.
- Tomada de decisão dinâmica: o fluxo de execução pode ser alterado conforme a entrada do usuário ou o contexto, permitindo que a aplicação escolha um assistente especializado que é mais adequado para a tarefa específica naquele momento.
- Gerenciamento avançado de estado e rotas: em vez de um fluxo rígido, temos um grafo onde é possível rotear a consulta para diferentes caminhos antes de chegar a uma resposta final, trazendo uma modularidade muito maior para aplicações que exigem respostas mais precisas e contextualizadas.

Considerando o agente "assistente de viagens", com o LangGraph, esse agente pode ser composto por múltiplos _prompts_ especializados, como por exemplo. Com essa arquitetura, o agente pode direcionar o fluxo de processamento conforme a necessidade, escolhendo o _prompt_ especializado mais adequado para executar uma tarefa e retornar a resposta.

Para ilustrar esse novo paradigma, vamos implementar um fluxo não linear em que temos dois _prompts_ especializados: um no tema **praias** e outro em **montanhas**. A partir da pergunta do usuário, será definido qual _prompt_ usar para responder a pergunta.

## Implementando fluxo não linear manualmente

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
api_key       = os.getenv("OPENAI_API_KEY")
model_version = os.getenv("OPENAI_MODEL_VERSION")

llm = ChatOpenAI(
    model=model_version,
    api_key=api_key
)


# define template dos prompts especializados
prompt_beach = ChatPromptTemplate.from_messages([
    ("system",      "Apresente-se como Sra Praia. Você é uma especialista em viagens com destinos para praia."),
    ("placeholder", "{conversation_historic}"),
    ("human",       "{query}")
])

prompt_mountain = ChatPromptTemplate.from_messages([
    ("system",      "Apresente-se como Sr Montanha. Você é uma especialista em viagens com destinos para montanhas e atividades radicais."),
    ("placeholder", "{conversation_historic}"),
    ("human",       "{query}")
])


# define cadeias especializadas
chain_beach    = prompt_beach    | llm | StrOutputParser()
chain_mountain = prompt_mountain | llm | StrOutputParser()

Primeiro vamos implementar um fluxo não linear "na mão", sem utilizar o LangGraph. Esse fluxo precisa começar por um "_prompt_ roteador", que recebe a pergunta do usuário e decide qual o caminho mais adequado, qual a melhor cadeia para lidar com essa pergunta.

Para garantir que a resposta dessa "cadeia roteadora" seja algo que permita definir o melhor caminho a ser seguido, vamos definir uma classe `Route` para representar a estrutura do dado que esperamos receber como resposta:

In [ ]:
from typing import TypedDict, Literal

# define output parser que sera usado com o prompt roteador
class Route(TypedDict):
    destino: Literal["praia", "montanha"]

# define template do prompt roteador
prompt_router = ChatPromptTemplate.from_messages([
    ("system", "Responda apenas com 'praia' ou 'montanha'"),
    ("human", "{query}")
])

Então definimos a cadeia que será responsável por tratar a pergunta inicial do usuário, passando para o _prompt_ roteador:

In [ ]:
# define cadeia roteadora
chain_router   = prompt_router | llm | Route
response_route = chain_router.invoke({"query": "Quero surfar em um lugar quente"})

print(type(response_route))
response_route

<class 'dict'>


{'content': 'praia',
 'additional_kwargs': {'refusal': None},
 'response_metadata': {'token_usage': {'completion_tokens': 27,
   'prompt_tokens': 32,
   'total_tokens': 59,
   'completion_tokens_details': {'accepted_prediction_tokens': 0,
    'audio_tokens': 0,
    'reasoning_tokens': 16,
    'rejected_prediction_tokens': 0,
    'text_tokens': None},
   'prompt_tokens_details': {'audio_tokens': 0,
    'cache_write_tokens': 0,
    'cached_tokens': 0,
    'image_tokens': None,
    'text_tokens': None}},
  'model_provider': 'openai',
  'model_name': 'gpt-6-luna',
  'system_fingerprint': None,
  'id': 'chatcmpl-EcUbZB4R71gEcYBf0ik7SDze0LAvN',
  'service_tier': 'default',
  'finish_reason': 'stop',
  'logprobs': None},
 'type': 'ai',
 'name': None,
 'id': 'lc_run--01a16dc2-5933-77a9-a739-f124d0d1030b-0',
 'tool_calls': [],
 'invalid_tool_calls': [],
 'usage_metadata': {'input_tokens': 32,
  'output_tokens': 27,
  'total_tokens': 59,
  'input_token_details': {'audio': 0, 'cache_read': 0, 'ca

Note que ao chamar `invoke` na cadeia roteadora, recebemos como retorno um objeto composto. Isso acontece porque a classe `Route` é apenas o **_schema_** (contrato) que descreve o formato do dado desejado, ela não implementa a interface de um `Runnable` ou de um `OutputParser`.

Ao usar `|` para encadear um objeto como componente da cadeia (logo após `llm`), é esperado que esse objeto possua um método chamado `invoke` (ou `parse`), que saibe exatamente como acessar a mensagem que o modelo de linguagem gerou e extrair apenas o conteúdo em texto (chave `content`). E a classe `Route` é apenas uma estrutura de dados (um `TypedDict` ou uma classe vazia), não possuindo os métodos necessários para processar o fluxo do LangChain. 

Então precisamos mudar a forma como definimos a cadeia, e fazemos isso com o método [`with_structured_output`](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel/with_structured_output), que vai usar a clase `Route` como um modelo para criar um _parser_ customizado que receberá a resposta do modelo de linguagem.

O método `with_structured_output(Route)`, faz três coisas importantes que o simples encadeamento (`|`) não faz:

- configura o modelo (via parâmetros da API, como o *Function Calling* ou *JSON Mode*) para que o modelo force a saída em um formato estruturado.
- injeta as definições de _schema_ (a classe `Route`) no contexto do modelo para que ele entenda exatamente o formato da resposta esperada.
- configura um _parser_ automático nos bastidores, e quando o modelo retorna a resposta, esse _parser_ pega o JSON/dado estruturado que veio da API e o converte para o formato definido na classe `Route`.

Seguindo essa abordagem, não é necessário instanciar manualmente um _parser_, e a cadeia usa a definição de `Route` para "moldar" a saída do modelo, convertendo a resposta em um dicionário Python que respeita as restrições impostas.

Se não usamos o `with_structured_output`, o modelo devolve apenas texto puro. E se quiséssemos usar a classe `Route` como um _parser_ sem o `with_structured_output`, seria necessário fazer a classe herdando de `BaseOutputParser` e implementar o método `parse` para converter manualmente para um dicionário a _string_ resultaante do modelo de linguagem.

In [23]:
from langchain_core.globals import set_debug

set_debug(True)
set_debug(False)

chain_router   = prompt_router | llm.with_structured_output(Route)
response_route = chain_router.invoke({"query": "Quero surfar em um lugar quente"})

print(type(response_route))
response_route

<class 'dict'>


{'destino': 'praia'}

E por fim criamos uma função que encapsula essa dinâmica, recebendo a pergunta inicial do usuário. Note que como fizemos a resposta da cadeia roteadora se adequar a definição da classe `Route`, podemos acessar a chave `'destino'` na resposta do método `invoke`.

In [17]:
def manual_response(user_query: str):
    route = chain_router.invoke({"query": user_query})["destino"]
    if route == "praia":
        return chain_beach.invoke({"query": user_query})
    return chain_mountain.invoke({"query": user_query})

In [10]:
print(manual_response("Quero passear em uma praia no Brasil"))

Olá! Sou a **Sra. Praia**, especialista em viagens para destinos litorâneos. 🏖️

No Brasil, algumas ótimas opções são:

- **Porto de Galinhas (PE):** piscinas naturais e passeios de jangada.
- **Maragogi (AL):** águas cristalinas e clima tranquilo.
- **Florianópolis (SC):** praias variadas, trilhas e vida noturna.
- **Jericoacoara (CE):** dunas, lagoas e um pôr do sol inesquecível.
- **Ubatuba (SP):** muitas praias cercadas pela Mata Atlântica.

Para eu indicar a melhor para você: **de onde vai sair, em que época pretende viajar e prefere sossego, aventura ou agito?**


In [19]:
print(manual_response("Quero fazer um passeio radical na natureza"))

Olá! Sou o Sr. Montanha, especialista em aventuras na natureza. 🏔️

Que tipo de emoção você procura: **rafting, rapel, tirolesa, escalada, trilha desafiadora** ou **mountain bike**? E em que cidade ou região você está? Com isso, posso sugerir um passeio que combine com seu nível de experiência e disposição.


In [24]:
print(manual_response("Recentemente escalei o Pico do Paraná e agora quero mais experiências similares"))

Sou o **Sr. Montanha**, especialista em aventuras de altitude e atividades radicais. Parabéns pelo Pico do Paraná — é uma conquista e tanto! Se você curtiu a combinação de esforço físico, trilha longa e visual de montanha, estas podem ser ótimas próximas experiências:

- **Itapiroca (PR)** — uma boa continuação na região do Pico do Paraná, com trilha exigente e vista ampla da serra.
- **Conjunto Marumbi (PR)** — mais íngreme e com trechos que pedem atenção; uma opção para quem quer terreno mais rochoso e desafiador.
- **Travessia Petrópolis–Teresópolis (RJ)** — aventura de vários dias pela Serra dos Órgãos, com trechos expostos e logística de pernoite.
- **Agulhas Negras (RJ)** — subida com trechos de escalaminhada e exposição; vale ir com guia se você não tiver experiência nesse tipo de terreno.
- **Travessia da Serra Fina (MG/SP/RJ)** — desafio de resistência em vários dias, com planejamento cuidadoso de água, clima e acampamento.

**Minha sugestão:** Serra Fina se você quer testar r